# Directories: named folders for organizing work

A record is addressed by its key: its directory name is the hash of what produced it. A
**directory** is addressed by its name: you choose where it is, as with datamate's
`Directory`. `td.Record` is a subclass of `td.Directory`, so both read and write files the same
way; they differ in how they are found and what their metadata holds:

| | `td.Directory` | `td.Record` |
|---|---|---|
| address | a path you name | `<root>/<Type>/<key[:16]>`, from the inputs and settings |
| metadata | typed fields in `_meta.yaml`, editable, tracked by git | spec, status, fields, provenance in `_meta.json` |
| two with the same settings | two directories: names are identities | one record |
| use it for | organizing work: questions, studies, versions | results: fits, tables, trained networks |

This notebook views a plain folder, declares a typed tree of research tracks, studies and
versions, moves and archives versions, links records to the version that produced them, and
saves figures into a version.

In [ ]:
import json
import tempfile
from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd
from jaxtyping import Float
from PIL import Image, ImageDraw

import tensordir as td

tmp = Path(tempfile.mkdtemp(prefix="tensordir-directories-"))
td.set_root(tmp / "records")  # the shared root of content-addressed records

## Any folder

`td.Directory(path)` views a folder. Opening it writes nothing. `load` and `record["name"]`
read a file by its suffix: JSON and YAML as values, CSV and parquet as DataFrames, `.npy` as
arrays, images and videos as objects that display themselves in Jupyter and marimo.
`td.register_suffix` adds a reader or writer for another suffix.

In [ ]:
folder = tmp / "downloads"
folder.mkdir()
(folder / "notes.txt").write_text("hello")
(folder / "settings.json").write_text(json.dumps({"fps": 30}))

downloads = td.Directory(folder)
print(downloads.keys(), downloads.load("settings.json"), downloads["notes.txt"])
print("files after opening:", sorted(p.name for p in folder.iterdir()))
downloads

## A typed tree

A `Directory` subclass declares metadata fields (annotations with defaults) and child types
(nested `Directory` classes). Each nested class becomes an accessor on its parent, named in
snake_case: `Track` becomes `dev.track`. `names = "v{n}"` tells `new()` how to name the next
child. `td.NodeId` marks a field whose values must name existing directories.

Define the tree in a module that every process imports: a reopened folder returns its
registered type, so a process that has not imported the class sees a plain `Directory`.

In [ ]:
class Dev(td.Directory):
    class Track(td.Directory):
        title: str = ""

        class Study(td.Directory):
            title: str = ""

            class Version(td.Directory):
                names = "v{n}"
                title: str = ""
                status: Literal["active", "superseded", "private"] = "active"
                builds_on: list[td.NodeId] = []


dev = Dev(tmp / "dev")
print("exists before the first write:", dev.exists)

`level.new(name, **fields)` creates a child with its metadata; without a name, it uses the
`names` pattern.

In [ ]:
track = dev.track.new("toy-model", title="A toy model")
inverse = track.study.new("inverse", title="Infer weights from responses")
v1 = inverse.version.new(title="the simplest inverse model")
v2 = inverse.version.new(title="rebuilt on the second data release", builds_on=["toy-model/inverse/v1"])
strf = track.study.new("strf")
strf.version.new(title="full-resolution STRFs")
print(v2.name, v2.id)
dev.tree(depth=4)

## Navigate

`level(name)` opens one child; an int is a position in `level.keys()`, so `-1` is the last.
A level reached across several parents (`dev.track.study.version`) lists IDs relative to where
the chain started. `resolve(id)` follows an ID from the directory it is called on.

In [ ]:
print(dev.track("toy-model").study("inverse").version("v2") == v2)
print(dev.track(0).study("inverse").version(-1) == v2)
print(dev.track.study.keys())
print(dev.track.study.version.keys())
print(dev.resolve("toy-model/inverse/v1") == v1, type(v1).__qualname__)

## Metadata

Fields are attributes. Setting one validates it against its annotation and writes
`_meta.yaml`: the type, the fields in declaration order and the aliases, without timestamps or
hosts, so the file can be tracked by git. Each write takes a short lease, so processes don't
overwrite each other.

In [ ]:
v1.status = "superseded"
print((v1.path / "_meta.yaml").read_text())

for bad in ({"status": "done"}, {"builds_on": ["toy-model/inverse/v9"]}, {"title": 3}):
    try:
        v1.update(**bad)
    except (ValueError, KeyError, TypeError) as exc:
        print(type(exc).__name__, exc)

## Move, archive, aliases

- `move(id)` renames the folder and adds the old ID to `aliases`, so the old ID still resolves.
- `archive()` moves a directory into its parent's `archive/` folder. Its ID stays the same,
  `resolve` and `level(name)` still find it, and `keys()` leaves it out.
- `new()` takes one more than the highest number used by a child, an archived child or an
  alias, so a number is never reused.

In [ ]:
track.study.new("ablations")
v2.move("toy-model/ablations/v1")
print(v2.id, v2.aliases, dev.resolve("toy-model/inverse/v2") == v2)

v1.archive()
print(v1.path.relative_to(dev.path), v1.id, v1.archived)
print(inverse.version.keys(), inverse.version.keys(archived=True))
print("next version:", inverse.version.new(title="a third design").name)

Aliases also carry legacy names, for example the flat version numbers used before a tree
existed.

In [ ]:
v1.aliases = ["v29"]
print(dev.resolve("v29") == v1)

## Which work produced a record

Records live in the shared root, so a version does not contain them. Inside
`with version.active():`, every record built stores the version as `producer` in its
provenance, including records built by subprocesses started inside the block, which inherit
`$TENSORDIR_PRODUCER`. The producer is not part of the key, so moving a version changes no key.

In [ ]:
class Fit(td.Record):
    loss: Float[np.ndarray, "n_steps"]

    def __init__(self, /, *, seed: int = 0):
        rng = np.random.default_rng(seed)
        self.loss = np.exp(-np.arange(40) / 10) + 0.02 * rng.random(40)


v3 = inverse.version("v3")
with v3.active():
    fit = Fit(seed=4)
print(fit.meta["provenance"]["producer"])
print(Fit(seed=5).meta["provenance"]["producer"])  # built outside the block

## Figures and tables

A version's figures and tables are files in its folder: `save` writes them by suffix (a
matplotlib figure, a PIL image or bytes for images; a DataFrame for `.csv`), and `load` reads
them back. An image loaded at the end of a cell displays itself.

Figures are not records: a figure's key would cover its data but not the plotting code, so a
changed plot would reopen the stale figure.

In [ ]:
def plot_loss(loss, size=(320, 160)):
    """A minimal line plot of `loss` as a PIL image."""
    image = Image.new("RGB", size, "white")
    x = np.linspace(10, size[0] - 10, len(loss))
    y = size[1] - 10 - (loss - loss.min()) / np.ptp(loss) * (size[1] - 20)
    ImageDraw.Draw(image).line(list(zip(x, y)), fill=(31, 119, 180), width=2)
    return image


v3.save("results/figures/loss.png", plot_loss(fit.loss))
v3.save("results/summary.csv", pd.DataFrame({"seed": [4], "final_loss": [float(fit.loss[-1])]}))
print(v3.load("results/summary.csv"))
v3.load("results/figures/loss.png")

## On disk

The tree shows each folder's type, read from its `_meta.yaml`, and the files saved into it; the
records sit in the shared root.

In [ ]:
td.Directory(tmp).tree(depth=7)